# DATA Processing

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = "/kaggle/input/sports-classification"

train_dir = os.path.join(data_dir, "train")
valid_dir = os.path.join(data_dir, "valid") 
test_dir  = os.path.join(data_dir, "test")

print("Imports complete. Data directories:")
print(f"Training data: {train_dir}")
print(f"Validation data: {valid_dir}")
print(f"Testing data: {test_dir}")
print(f"Using device: {device}")


Imports complete. Data directories:
Training data: /kaggle/input/sports-classification/train
Validation data: /kaggle/input/sports-classification/valid
Testing data: /kaggle/input/sports-classification/test
Using device: cuda


## Load + Explore Dataset

Seeing what sports are present and how many images we have

In [2]:
# List all sport classes in the training directory
sports = sorted(os.listdir(train_dir))

# Count images in each sport class
image_counts = {}
for sport in sports:
    sport_path = os.path.join(train_dir, sport)
    if os.path.isdir(sport_path):
        image_count = len(os.listdir(sport_path))
        image_counts[sport] = image_count

# Display statistics
print(f"Total number of sports: {len(sports)}")
print(f"Total training images: {sum(image_counts.values())}")
print(f"\nImages per sport (first 10):")
for i, (sport, count) in enumerate(sorted(image_counts.items())[:10]):
    print(f"  {sport}: {count} images")


Total number of sports: 100
Total training images: 13493

Images per sport (first 10):
  air hockey: 112 images
  ampute football: 112 images
  archery: 132 images
  arm wrestling: 99 images
  axe throwing: 113 images
  balance beam: 147 images
  barell racing: 123 images
  baseball: 174 images
  basketball: 169 images
  baton twirling: 108 images


## Preprocess Training Data

In [3]:
# Create a mapping from sport name to numeric label
sport_to_label = {sport: idx for idx, sport in enumerate(sports)}
label_to_sport = {idx: sport for sport, idx in sport_to_label.items()}

for sport, label in list(sport_to_label.items())[:5]:
    print(f"  {label}: {sport}")

  0: air hockey
  1: ampute football
  2: archery
  3: arm wrestling
  4: axe throwing


In [4]:
train_images = []
train_labels = []

for sport in sports:
    sport_path = os.path.join(train_dir, sport)
    sport_label = sport_to_label[sport]
    
    # Load each image in this sport's folder
    for image_file in os.listdir(sport_path):
        img_path = os.path.join(sport_path, image_file)
        try:
            img = Image.open(img_path).convert('RGB')
            # Convert to numpy array 
            img_array = np.array(img)
            # Store image and its label
            train_images.append(img_array) # add to list
            train_labels.append(sport_label) # add to list
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

# Convert lists to numpy arrays (easy for NN later)
train_images = np.array(train_images)
train_labels = np.array(train_labels)

print(f"Shape of training images array: {train_images.shape}")
print(f"Shape of training labels array: {train_labels.shape}")

Error loading /kaggle/input/sports-classification/train/high jump/159.lnk: cannot identify image file '/kaggle/input/sports-classification/train/high jump/159.lnk'
Shape of training images array: (13492, 224, 224, 3)
Shape of training labels array: (13492,)


## CNN model

My own CNN from Scratch

In [5]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes, dropout_p=0.3):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)   # 224 -> 112

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)   # 112 -> 56

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)   # 56 -> 28

        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()

        
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))   

        x = self.gap(x)                             
        x = self.flatten(x)                          

        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))
        x = self.dropout(x)

        x = self.fc3(x)
        return x


# Create the model
num_classes = len(train_dataset.classes)
model = SimpleCNN(num_classes=num_classes, dropout_p=0.5).to(device)

print("Device:", next(model.parameters()).device)


NameError: name 'train_dataset' is not defined

Pretrained EfficientNet CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms, datasets


# Parameters
learning_rate = 3e-4
batch_size = 32
num_epochs = 20
weight_decay = 1e-4

# Transforms for EfficientNet
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# Data sets
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform_train)
val_dataset   = datasets.ImageFolder(root=valid_dir, transform=transform_eval)
test_dataset  = datasets.ImageFolder(root=test_dir,  transform=transform_eval)


# DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=2, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=2, pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)


# EfficientNet-B0 model
num_classes = len(train_dataset.classes)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)


# Loss + Optimizer
loss_function = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print("EfficientNet training setup complete:")
print("  Model:", "efficientnet_b0 (pretrained)")
print("  Num classes:", num_classes)
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {num_epochs}")
print(f"  LR: {learning_rate}")
print(f"  Weight decay: {weight_decay}")
print(f"  Batches/epoch: {len(train_loader)}")
print("  Device:", next(model.parameters()).device)


In [ ]:
# ONLY train classifier in pretrained model
for p in model.features.parameters():
    p.requires_grad = False

optimizer = optim.AdamW(model.classifier.parameters(), lr=3e-4, weight_decay=1e-4)

training_losses, training_accuracies = [], []
val_losses, val_accuracies = [], []

patience = 5
best_val_acc = 0.0
epochs_no_improve = 0
best_model_state = None

print("Starting training (classifier-only)...")

for epoch in range(num_epochs):

    # Train
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        preds = model(batch_images)
        loss = loss_function(preds, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_images.size(0)
        correct += (preds.argmax(dim=1) == batch_labels).sum().item()
        total += batch_labels.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    training_losses.append(train_loss)
    training_accuracies.append(train_acc)

    # Validate
    model.eval()
    v_loss_sum = 0.0
    v_correct = 0
    v_total = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)

            v_preds = model(val_images)
            v_loss = loss_function(v_preds, val_labels)

            v_loss_sum += v_loss.item() * val_images.size(0)
            v_correct += (v_preds.argmax(dim=1) == val_labels).sum().item()
            v_total += val_labels.size(0)

    val_loss = v_loss_sum / v_total
    val_acc = v_correct / v_total

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Early Stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered. Best Val Acc: {best_val_acc:.4f}")
            break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
model.to(device)

print("Training complete! Best Val Acc:", best_val_acc)
print("History lengths:",
      len(training_losses), len(training_accuracies),
      len(val_losses), len(val_accuracies))


In [ ]:
# Test
model.load_state_dict(best_model_state)
model.to(device)
model.eval()


test_correct = 0
test_total = 0
test_loss_sum = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = loss_function(outputs, labels)

        test_loss_sum += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

print("Test Acc:", test_correct / test_total)
print("Test Loss:", test_loss_sum / test_total)


In [ ]:
# Safety checks
print("len(training_losses):", len(training_losses))
print("len(training_accuracies):", len(training_accuracies))
print("len(val_losses):", len(val_losses))
print("len(val_accuracies):", len(val_accuracies))

# If lists are empty, stop here with a clear message
if len(training_accuracies) == 0 or len(val_accuracies) == 0:
    raise ValueError(
        "Your training/validation history lists are empty. "
        "Make sure your training loop appends to training_losses/training_accuracies "
        "and val_losses/val_accuracies each epoch, and re-run the training cell before plotting."
    )

# Now safe to plot/print
epochs = range(1, len(training_losses) + 1)

import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, training_losses, linewidth=2, label="Train Loss")
ax1.plot(epochs, val_losses, linewidth=2, label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss Over Epochs")
ax1.grid(True); ax1.legend()

ax2.plot(epochs, training_accuracies, linewidth=2, label="Train Acc")
ax2.plot(epochs, val_accuracies, linewidth=2, label="Val Acc")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_title("Accuracy Over Epochs")
ax2.grid(True); ax2.legend()

plt.tight_layout()
plt.show()

print(f"Final Train Accuracy (last epoch): {training_accuracies[-1]:.4f}")
print(f"Best Val Accuracy (max): {max(val_accuracies):.4f}")
